In [89]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [52]:
# data=pd.read_csv('email_classification.csv')
data=pd.read_csv('spam_Emails_data.csv')
data.head()

,label,text
0,Spam,viiiiiiagraaaa\nonly for the ones that want to...
1,Ham,got ice thought look az original message ice o...
2,Spam,yo ur wom an ne eds an escapenumber in ch ma n...
3,Spam,start increasing your odds of success & live s...
4,Ham,author jra date escapenumber escapenumber esca...


In [53]:
temp=pd.read_csv('email_classification.csv')
temp=temp.rename(columns={'email': 'text'}).replace({'label': {'spam': 'Spam'}})
data=pd.concat((data, temp))

In [54]:
del temp

In [55]:
data.dropna(inplace=True)

In [90]:
# X, y=data['email'], data['label']
X, y=data['text'], data['label']
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.25)
# count_vec=CountVectorizer(stop_words='english', max_features=100000)
count_vec=TfidfVectorizer(stop_words='english', max_features=100000)
count_vec.fit(X_train, y_train)

TfidfVectorizer(max_features=100000, stop_words='english')

In [91]:
# X_train_=count_vec.transform(X_train).toarray()
X_train_=count_vec.transform(X_train)
y_train_=y_train.apply(lambda x: int(x=='Spam'))
# X_test_=count_vec.transform(X_test).toarray()
X_test_=count_vec.transform(X_test)
y_test_=y_test.apply(lambda x: int(x=='Spam'))

In [92]:
X_train_

<145521x100000 sparse matrix of type '<class 'numpy.float64'>'
	with 13185886 stored elements in Compressed Sparse Row format>

In [93]:
class NaiveBayes:
    def __init__(self):
        self.p=0
        self.p1=None
        self.p0=None
    
    def estimate_params(self, X, y):
        subset_index=(y==0)
        return np.array((X[subset_index, :]>0).mean(axis=0))[0], np.array((X[~subset_index, :]>0).mean(axis=0))[0]

    def add_smoothing(self, X, y):
        n_features=X.shape[1]
        empty=sparse.csr_matrix(0, shape=(1, n_features))
        full=sparse.csr_matrix(1, shape=(1, n_features))
        new_data_matrix=sparse.vstack((X, empty, empty.copy(), full, full.copy()))
        new_labels=np.concatenate((y, [0, 1, 0, 1]))
        return new_data_matrix, new_labels

    def fit(self, X, y):
        X_new, y_new=self.add_smoothing(X, y)
        self.p=(y_new==1).mean()
        self.p0, self.p1=self.estimate_params(X_new, y_new)
        self.p0[self.p0==0]=1e-6
        self.p1[self.p1==0]=1e-6
        return self
    
    def predict(self, X):
        X_new=(X>0).astype(int)
        predictions= X_new @ np.log( (self.p1 * (1-self.p0))  /  (self.p0 * (1-self.p1)) )
        predictions+= np.sum( np.log( (1-self.p1) / (1-self.p0) ) ) + np.log( self.p/(1-self.p) )
        return (predictions>=0).astype(int)


In [94]:
nb=NaiveBayes()
nb.fit(X_train_, y_train_)

In [95]:
(nb.p1==0).sum()

0

In [96]:
predictions=nb.predict(X_train_)
(predictions==y_train_).mean()

0.900268689742374

In [97]:
predictions=nb.predict(X_test_)
(predictions==y_test_).mean()

0.8973571369670982

# Logistic Regression

In [98]:
class LogReg:
    def __init__(self, iterations, step_size):
        self.iterations=iterations
        self.step_size=step_size
        self.coef_=None
        self.errors=[]
    
    def sigmoid(self, z):
        return 1/(1+ np.exp(-z))

    def loss(self, X, y):
        probs= self.sigmoid(X @ self.coef_)
        return -(y*np.log(probs) + (1-y)*np.log(1-probs)).mean()

    def gradient(self, X, y):
        y_hat=self.sigmoid(X @ self.coef_)
        differences=y_hat-y
        return np.array((sparse.diags(differences) @ X).sum(axis=0))[0]
    
    def gradient_descent(self, X, y):
        self.coef_=np.zeros(X.shape[1])
        for i in range(self.iterations):
            grad=self.gradient(X, y)
            self.coef_-=self.step_size*grad
            self.errors.append(self.loss(X, y))

    def normalize(self, X):
        return (X@sparse.diags((np.array(X.power(2).sum(axis=0))[0]+1)**-0.5))

    def fit(self, X, y):
        self.gradient_descent(self.normalize(X), y)
    
    def predict(self, X):
        probs=self.normalize(X) @ self.coef_
        return (probs>=0).astype(int)        
        

In [99]:
log_reg=LogReg(30, 0.001)
log_reg.fit(X_train_, y_train_.values)

In [100]:
predictions=log_reg.predict(X_train_)
(predictions==y_train_).mean()

0.9552504449529621

In [101]:
predictions=log_reg.predict(X_test_)
(predictions==y_test_).mean()

0.9505030098128143

In [102]:
new_data=pd.read_csv('email_classification.csv')
new_data.dropna(inplace=True)
X_new=count_vec.transform(new_data['email'])
y_new=new_data['label'].apply(lambda x: int(x=='spam'))

In [103]:
predictions=log_reg.predict(X_new)
(predictions==y_new).mean()

0.7150837988826816

In [104]:
predictions=nb.predict(X_new)
(predictions==y_new).mean()

0.441340782122905